<div style="background:linear-gradient(135deg,#1e3a8a 0%,#1e40af 60%,#2563eb 100%);padding:36px 32px;border-radius:14px;color:white;font-family:'Segoe UI',sans-serif;">
  <h1 style="margin:0 0 10px 0;font-size:2.1em;letter-spacing:1px;">🏢 Employee Attrition & Workforce Analysis</h1>
  <p style="margin:0 0 6px 0;font-size:1.1em;opacity:0.92;">IBM SkillsBuild Data Analytics with AI Academic Internship Program</p>
  <p style="margin:0;font-size:0.97em;opacity:0.80;">BharatCares in association with AICTE &nbsp;|&nbsp; Student: <strong>Anish Kumar</strong></p>
</div>

---
## 📌 Problem Statement
Employee attrition — the voluntary or involuntary departure of employees — is a significant challenge for organizations. High attrition rates lead to increased recruitment and training costs, loss of institutional knowledge, and reduced team productivity. This project analyzes the IBM HR Analytics dataset to uncover patterns in attrition across departments, job roles, age, overtime, travel, salary, and satisfaction — and delivers actionable HR recommendations.

---
## 🎯 Objectives
1. Load and explore the IBM HR Analytics dataset  
2. Perform data quality checks and clean the data  
3. Engineer useful derived features  
4. Calculate key business metrics  
5. Conduct EDA across 15+ dimensions  
6. Create professional data visualizations  
7. Perform correlation analysis  
8. Derive key insights and business recommendations

---
## 📦 Dataset
| Attribute | Value |
|---|---|
| **Source** | IBM HR Analytics Employee Attrition & Performance — Kaggle |
| **URL** | https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset |
| **Records** | 1,470 employees |
| **Features** | 35 columns |
| **Target** | `Attrition` (Yes / No) |

---
## 🛠️ Technologies Used
| Library | Purpose |
|---|---|
| **Python 3** | Core programming language |
| **Pandas** | Data loading, cleaning, groupby analysis |
| **NumPy** | Numerical computations |
| **Matplotlib** | Bar charts, histograms, pie charts |
| **Seaborn** | Statistical visualizations, heatmap |
| **Jupyter Notebook** | Interactive development environment |

---
## 1️⃣ Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Plot style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 110,
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
})
sns.set_style('whitegrid')

COLOR_YES     = '#dc2626'
COLOR_NO      = '#2563eb'
AVG_RATE      = 16.12
AVG_COLOR     = '#64748b'

print("✅ Libraries imported successfully.")

---
## 2️⃣ Load Dataset

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
print(f"✅ Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 3️⃣ Data Overview

In [ ]:
print("=== First 5 Rows ===")
df.head()

In [ ]:
print("=== Last 5 Rows ===")
df.tail()

In [ ]:
print(f"Rows : {df.shape[0]:,}")
print(f"Cols : {df.shape[1]}")
print("\nColumn list:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:>2}. {col}")

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
df.describe(include='object')

> **Interpretation:** 1,470 rows, 35 columns. Mix of numerical and categorical variables. No obvious anomalies at first glance.

---
## 4️⃣ Data Quality Check

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║         DATA QUALITY REPORT              ║")
print("╠══════════════════════════════════════════╣")
print(f"║  Missing values  : {df.isnull().sum().sum():<23}║")
print(f"║  Duplicate rows  : {df.duplicated().sum():<23}║")
num_neg = sum((df[c]<0).sum() for c in df.select_dtypes(include=np.number).columns)
print(f"║  Negative values : {num_neg:<23}║")
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f"║  Constant columns: {str(const_cols):<23}║")
print("╚══════════════════════════════════════════╝")

In [ ]:
# Categorical unique values
print("Unique values in categorical columns:")
for col in df.select_dtypes(include='object').columns:
    print(f"  {col:25}: {df[col].unique().tolist()}")

In [ ]:
# Outlier check on MonthlyIncome
Q1, Q3 = df['MonthlyIncome'].quantile([0.25, 0.75])
IQR = Q3 - Q1
outliers = df[(df['MonthlyIncome'] < Q1-1.5*IQR) | (df['MonthlyIncome'] > Q3+1.5*IQR)]
print(f"MonthlyIncome outliers (IQR): {len(outliers)} rows")
print(f"  Range: ${df['MonthlyIncome'].min():,} – ${df['MonthlyIncome'].max():,}")
print("  → Retained: high-income outliers represent valid senior roles (Manager, Research Director)")

> **Summary:** No missing values or duplicates. Three constant columns (`EmployeeCount`, `StandardHours`, `Over18`) add no information. All outliers are valid senior-role salaries.

---
## 5️⃣ Data Cleaning

In [ ]:
df_clean = df.copy()

# Drop constant and identifier columns
drop_cols = ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df_clean.drop(columns=drop_cols, inplace=True)

print(f"✅ Dropped columns : {drop_cols}")
print(f"   Original shape  : {df.shape}")
print(f"   Cleaned shape   : {df_clean.shape}")
print(f"   Rows retained   : {len(df_clean):,} (no rows removed)")
print(f"   Missing values  : {df_clean.isnull().sum().sum()}")

> **Cleaned dataset is ready.** All 1,470 employee records retained. 4 uninformative columns removed.

---
## 6️⃣ Feature Engineering

In [ ]:
# AgeGroup: bin Age into 5 generational bands
df_clean['AgeGroup'] = pd.cut(df_clean['Age'],
    bins=[18,25,35,45,55,65], labels=['18-25','26-35','36-45','46-55','56-65'])

# TenureGroup: career-stage tenure buckets
df_clean['TenureGroup'] = pd.cut(df_clean['YearsAtCompany'],
    bins=[-1,2,5,10,20,100], labels=['0-2 yrs','3-5 yrs','6-10 yrs','11-20 yrs','20+ yrs'])

# IncomeGroup: 4 salary bands
df_clean['IncomeGroup'] = pd.cut(df_clean['MonthlyIncome'],
    bins=[0,3000,6000,10000,20000], labels=['Low (<3K)','Mid (3-6K)','High (6-10K)','Very High (>10K)'])

# AttritionBinary: 0/1 for correlation analysis
df_clean['AttritionBinary'] = (df_clean['Attrition'] == 'Yes').astype(int)

print("✅ Derived features created:")
for f in ['AgeGroup','TenureGroup','IncomeGroup','AttritionBinary']:
    print(f"   {f:18}: {df_clean[f].value_counts(sort=False).to_dict()}")

> **AgeGroup** enables generational attrition comparison. **TenureGroup** reveals when in the career employees are most at risk. **IncomeGroup** links salary bands to attrition. **AttritionBinary** enables numerical correlation analysis.

---
## 7️⃣ Key Business Metrics

In [ ]:
total     = len(df_clean)
left      = (df_clean['Attrition']=='Yes').sum()
active    = total - left
attr_rate = round(left/total*100, 2)

from IPython.display import HTML

kpi_html = f"""
<div style="font-family:'Segoe UI',sans-serif;">
  <h3 style="color:#1e293b;border-bottom:3px solid #2563eb;padding-bottom:6px;">📊 Key Performance Indicators</h3>
  <div style="display:flex;flex-wrap:wrap;gap:14px;margin-top:14px;">
    <div style="background:#eff6ff;border-left:5px solid #2563eb;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#1d4ed8;">{total:,}</div>
      <div style="color:#64748b;font-size:0.93em;">Total Employees</div>
    </div>
    <div style="background:#fef2f2;border-left:5px solid #dc2626;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#b91c1c;">{left}</div>
      <div style="color:#64748b;font-size:0.93em;">Employees Who Left</div>
    </div>
    <div style="background:#f0fdf4;border-left:5px solid #16a34a;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#15803d;">{active:,}</div>
      <div style="color:#64748b;font-size:0.93em;">Active Employees</div>
    </div>
    <div style="background:#fffbeb;border-left:5px solid #d97706;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#b45309;">{attr_rate}%</div>
      <div style="color:#64748b;font-size:0.93em;">Attrition Rate</div>
    </div>
    <div style="background:#faf5ff;border-left:5px solid #7c3aed;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#6d28d9;">\${df_clean['MonthlyIncome'].mean():,.0f}</div>
      <div style="color:#64748b;font-size:0.93em;">Avg Monthly Income</div>
    </div>
    <div style="background:#ecfeff;border-left:5px solid #0891b2;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#0e7490;">{df_clean['Age'].mean():.1f}</div>
      <div style="color:#64748b;font-size:0.93em;">Average Age</div>
    </div>
    <div style="background:#fff1f2;border-left:5px solid #be123c;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#9f1239;">{df_clean['JobSatisfaction'].mean():.2f}/4</div>
      <div style="color:#64748b;font-size:0.93em;">Avg Job Satisfaction</div>
    </div>
    <div style="background:#f0fdf4;border-left:5px solid #15803d;padding:16px 24px;border-radius:8px;min-width:160px;">
      <div style="font-size:2em;font-weight:700;color:#166534;">{df_clean['YearsAtCompany'].mean():.1f}</div>
      <div style="color:#64748b;font-size:0.93em;">Avg Years at Company</div>
    </div>
  </div>
</div>
"""
display(HTML(kpi_html))

> **1 in 6 employees left.** Average satisfaction (2.73/4) and work-life balance (2.76/4) indicate moderate engagement issues across the workforce.

---
## 8️⃣ Exploratory Data Analysis
Helper function used throughout all EDA sections:

In [ ]:
def attrition_summary(df, col):
    """Return attrition count and rate grouped by `col`."""
    r = df.groupby(col, observed=True).agg(
        Total=('Attrition','count'),
        Left=('Attrition', lambda x: (x=='Yes').sum())
    ).reset_index()
    r['Rate(%)'] = (r['Left'] / r['Total'] * 100).round(2)
    return r

def add_bar_labels(ax, bars, fmt='{:.1f}%', offset=0.4):
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x()+bar.get_width()/2, h+offset, fmt.format(h),
                ha='center', va='bottom', fontsize=9, fontweight='bold')

def avg_line(ax, val=AVG_RATE, label=None):
    label = label or f'Company Avg ({val}%)'
    ax.axhline(val, color=AVG_COLOR, linestyle='--', linewidth=1.5, label=label)

print('✅ Helper functions ready.')

### A. Attrition by Department

In [ ]:
dept = attrition_summary(df_clean, 'Department')
print(dept.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,5))
colors = [COLOR_YES if v > AVG_RATE else COLOR_NO for v in dept['Rate(%)']]
bars = ax.bar(dept['Department'], dept['Rate(%)'], color=colors, edgecolor='white', width=0.5, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax)
red_p  = mpatches.Patch(color=COLOR_YES, label='Above average')
blue_p = mpatches.Patch(color=COLOR_NO,  label='Below average')
ax.legend(handles=[red_p, blue_p, plt.Line2D([0],[0],color=AVG_COLOR,linestyle='--',linewidth=1.5)],
          labels=['Above average','Below average',f'Avg {AVG_RATE}%'], fontsize=9)
ax.set_title('Attrition Rate by Department', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Department', fontsize=11); ax.set_ylabel('Attrition Rate (%)', fontsize=11)
ax.set_ylim(0, 30)
plt.tight_layout(); plt.show()

> **Sales (20.63%)** and **Human Resources (19.05%)** both exceed the company average. **Research & Development** is the largest department but has the lowest rate (13.84%).

### B. Attrition by Job Role

In [ ]:
role = attrition_summary(df_clean, 'JobRole').sort_values('Rate(%)', ascending=False)
print(role.to_string(index=False))

fig, ax = plt.subplots(figsize=(11,6))
role_s = role.sort_values('Rate(%)', ascending=True)
colors_r = [COLOR_YES if v > AVG_RATE else COLOR_NO for v in role_s['Rate(%)']]
bars = ax.barh(role_s['JobRole'], role_s['Rate(%)'], color=colors_r, edgecolor='white', height=0.6, zorder=3)
for bar, val in zip(bars, role_s['Rate(%)']):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')
ax.axvline(AVG_RATE, color=AVG_COLOR, linestyle='--', linewidth=1.5, label=f'Avg {AVG_RATE}%')
ax.set_title('Attrition Rate by Job Role', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Attrition Rate (%)', fontsize=11); ax.set_xlim(0,50); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

> **Sales Representatives (39.76%)** — nearly 2 in 5 leave. Senior roles (Manager: 4.90%, Research Director: 2.50%) are well-retained.

### C. Attrition by Gender

In [ ]:
gender = attrition_summary(df_clean, 'Gender')
print(gender.to_string(index=False))

fig, axes = plt.subplots(1,2, figsize=(11,5))
colors_g = ['#ec4899','#3b82f6']
bars = axes[0].bar(gender['Gender'], gender['Rate(%)'], color=colors_g, edgecolor='white', width=0.4, zorder=3)
add_bar_labels(axes[0], bars)
avg_line(axes[0]); axes[0].legend(fontsize=9)
axes[0].set_title('Attrition Rate by Gender', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Attrition Rate (%)'); axes[0].set_ylim(0,25)
axes[1].pie(gender['Left'], labels=gender['Gender'], autopct='%1.1f%%', colors=colors_g,
           startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Share of Attrition by Gender', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

> Male employees (17.01%) have a slightly higher rate than female (14.80%). The difference is modest but consistent.

### D. Attrition by Age Group

In [ ]:
ag = attrition_summary(df_clean, 'AgeGroup')
print(ag.to_string(index=False))

fig, ax = plt.subplots(figsize=(10,5))
palette_age = ['#dc2626','#ea580c','#ca8a04','#16a34a','#2563eb']
bars = ax.bar(ag['AgeGroup'].astype(str), ag['Rate(%)'], color=palette_age, edgecolor='white', width=0.55, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
for bar, n in zip(bars, ag['Total']):
    ax.text(bar.get_x()+bar.get_width()/2, -3, f'n={n}', ha='center', fontsize=8, color='#64748b')
ax.set_title('Attrition Rate by Age Group', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Age Group'); ax.set_ylabel('Attrition Rate (%)')
ax.set_ylim(-5, 45)
plt.tight_layout(); plt.show()

> **18–25 year-olds** have the highest attrition at **34.78%**. Mid-career employees (36–45) are the most stable at 9.19%.

### E. Attrition by Overtime

In [ ]:
ot = attrition_summary(df_clean, 'OverTime')
print(ot.to_string(index=False))
print(f"\n28.3% of all employees work overtime ({(df_clean['OverTime']=='Yes').sum()} employees)")

fig, axes = plt.subplots(1,2, figsize=(12,5))
bars = axes[0].bar(ot['OverTime'], ot['Rate(%)'], color=[COLOR_NO, COLOR_YES],
                   edgecolor='white', width=0.45, zorder=3)
add_bar_labels(axes[0], bars, offset=0.8)
avg_line(axes[0]); axes[0].legend(fontsize=9)
axes[0].set_title('Attrition Rate: Overtime vs No Overtime', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Works Overtime'); axes[0].set_ylabel('Attrition Rate (%)'); axes[0].set_ylim(0,42)

ot_counts = df_clean.groupby(['OverTime','Attrition'], observed=True).size().unstack(fill_value=0)
ot_counts.plot(kind='bar', ax=axes[1], color=[COLOR_NO, COLOR_YES], edgecolor='white', rot=0)
axes[1].set_title('Headcount: Overtime vs Attrition', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Works Overtime'); axes[1].set_ylabel('Number of Employees')
axes[1].legend(['Stayed','Left'], fontsize=9)
for container in axes[1].containers:
    axes[1].bar_label(container, fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

> 🚨 **Overtime is the single strongest attrition driver.** Workers on overtime leave at **30.53%** — nearly **3× higher** than non-overtime workers (10.44%).

### F. Attrition by Business Travel

In [ ]:
bt = attrition_summary(df_clean, 'BusinessTravel').sort_values('Rate(%)', ascending=False)
print(bt.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(bt['BusinessTravel'], bt['Rate(%)'], color=[COLOR_YES,'#d97706',COLOR_NO],
              edgecolor='white', width=0.5, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Business Travel Frequency', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Travel Category'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,35)
plt.tight_layout(); plt.show()

> Frequent travelers leave at **24.91%** vs non-travelers at **8.00%** — a 3× difference.

### G. Attrition by Marital Status

In [ ]:
ms = attrition_summary(df_clean, 'MaritalStatus').sort_values('Rate(%)', ascending=False)
print(ms.to_string(index=False))

fig, ax = plt.subplots(figsize=(8,5))
bars = ax.bar(ms['MaritalStatus'], ms['Rate(%)'], color=[COLOR_YES, COLOR_NO, '#16a34a'],
              edgecolor='white', width=0.5, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Marital Status', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Marital Status'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,35)
plt.tight_layout(); plt.show()

> **Single employees (25.53%)** are more than 2× as likely to leave as divorced employees (10.09%).

### H. Attrition by Education Field

In [ ]:
ef = attrition_summary(df_clean, 'EducationField').sort_values('Rate(%)', ascending=False)
print(ef.to_string(index=False))

### I. Attrition by Job Level

In [ ]:
jl = attrition_summary(df_clean, 'JobLevel')
print(jl.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(jl['JobLevel'].astype(str), jl['Rate(%)'],
              color=['#dc2626','#ea580c','#ca8a04','#16a34a','#2563eb'],
              edgecolor='white', width=0.55, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Job Level', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Job Level  (1=Entry → 5=Senior)'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,35)
plt.tight_layout(); plt.show()

> Entry-level (Level 1): **26.34%** vs senior (Level 5): **7.25%**. Career progression is a powerful retention mechanism.

### J. Attrition by Tenure Group

In [ ]:
tg = attrition_summary(df_clean, 'TenureGroup')
print(tg.to_string(index=False))

### K. Attrition by Job Satisfaction

In [ ]:
js = attrition_summary(df_clean, 'JobSatisfaction')
print(js.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(js['JobSatisfaction'].astype(str), js['Rate(%)'],
              color=['#dc2626','#ea580c','#2563eb','#16a34a'],
              edgecolor='white', width=0.55, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Job Satisfaction Level', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Job Satisfaction (1=Low → 4=High)'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,32)
plt.tight_layout(); plt.show()

### L. Attrition by Work-Life Balance

In [ ]:
wlb = attrition_summary(df_clean, 'WorkLifeBalance')
print(wlb.to_string(index=False))

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar(wlb['WorkLifeBalance'].astype(str), wlb['Rate(%)'],
              color=['#dc2626','#ea580c','#2563eb','#16a34a'],
              edgecolor='white', width=0.55, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Work-Life Balance Rating', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Work-Life Balance (1=Bad → 4=Best)'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,42)
plt.tight_layout(); plt.show()

> WLB Level 1: **31.25%** attrition — over twice the level at rating 3 (14.22%).

### M & N. Average Monthly Income

In [ ]:
inc_dept = df_clean.groupby('Department')['MonthlyIncome'].mean().round(2).sort_values(ascending=False)
inc_role = df_clean.groupby('JobRole')['MonthlyIncome'].mean().round(2).sort_values(ascending=False)

print("Avg Monthly Income by Department:")
print(inc_dept.to_string())
print("\nAvg Monthly Income by Job Role:")
print(inc_role.to_string())

fig, axes = plt.subplots(1,2, figsize=(16,5))
id_sorted = inc_dept.sort_values(ascending=True)
colors_d = plt.cm.Blues(np.linspace(0.45,0.85,len(id_sorted)))
bars1 = axes[0].barh(id_sorted.index, id_sorted.values, color=colors_d, edgecolor='white', height=0.45)
for bar, val in zip(bars1, id_sorted.values):
    axes[0].text(bar.get_width()+120, bar.get_y()+bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')
axes[0].set_title('Avg Monthly Income by Department', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Average Monthly Income ($)'); axes[0].set_xlim(0,10500)

ir_sorted = inc_role.sort_values(ascending=True)
colors_r2 = plt.cm.Blues(np.linspace(0.3,0.9,len(ir_sorted)))
bars2 = axes[1].barh(ir_sorted.index, ir_sorted.values, color=colors_r2, edgecolor='white', height=0.6)
for bar, val in zip(bars2, ir_sorted.values):
    axes[1].text(bar.get_width()+150, bar.get_y()+bar.get_height()/2,
                 f'${val:,.0f}', va='center', fontsize=9, fontweight='bold')
axes[1].set_title('Avg Monthly Income by Job Role', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Average Monthly Income ($)'); axes[1].set_xlim(0,21000)
plt.tight_layout(); plt.show()

> Managers earn the most ($17,182/month); Sales Representatives earn the least ($2,626/month) — and have the highest attrition.

### O. Job Satisfaction by Department

In [ ]:
js_dept = df_clean.groupby('Department')['JobSatisfaction'].mean().round(2)
print(js_dept.to_string())

### P. Stayed vs Left — Profile Comparison

In [ ]:
comp = df_clean.groupby('Attrition')[['Age','MonthlyIncome','JobSatisfaction',
                                       'WorkLifeBalance','YearsAtCompany',
                                       'TotalWorkingYears','JobLevel']].mean().round(2)
print(comp)

attrs  = ['Age (yrs)','Monthly\nIncome ($)','Years at\nCompany','Total Working\nYears']
stayed = comp.loc['No',  ['Age','MonthlyIncome','YearsAtCompany','TotalWorkingYears']].values
left   = comp.loc['Yes', ['Age','MonthlyIncome','YearsAtCompany','TotalWorkingYears']].values
x = np.arange(len(attrs)); width = 0.35

fig, ax = plt.subplots(figsize=(12,5))
b1 = ax.bar(x-width/2, stayed, width, color=COLOR_NO,  edgecolor='white', label='Stayed')
b2 = ax.bar(x+width/2, left,   width, color=COLOR_YES, edgecolor='white', label='Left')
for bar, val in zip(b1, stayed):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+60,
            f'{val:,.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLOR_NO)
for bar, val in zip(b2, left):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+60,
            f'{val:,.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLOR_YES)
ax.set_title('Employee Profile: Stayed vs Left', fontsize=14, fontweight='bold', pad=12)
ax.set_xticks(x); ax.set_xticklabels(attrs, fontsize=10)
ax.set_ylabel('Value'); ax.legend(fontsize=10)
plt.tight_layout(); plt.show()

> Employees who left were younger (33.6 vs 37.6 yrs), paid less ($4,787 vs $6,833/month), had less experience and lower tenure — a consistent, statistically meaningful pattern.

---
## 9️⃣ Additional Visualizations

In [ ]:
# Distributions — Years at Company & Monthly Income
stayed_yrs = df_clean[df_clean['Attrition']=='No']['YearsAtCompany']
left_yrs   = df_clean[df_clean['Attrition']=='Yes']['YearsAtCompany']
bins_yr    = np.arange(0, 41, 2)

fig, axes = plt.subplots(1,2, figsize=(14,5))
axes[0].hist(stayed_yrs, bins=bins_yr, color=COLOR_NO,  alpha=0.7, edgecolor='white', label='Stayed')
axes[0].hist(left_yrs,   bins=bins_yr, color=COLOR_YES, alpha=0.7, edgecolor='white', label='Left')
axes[0].axvline(stayed_yrs.mean(), color=COLOR_NO,  linestyle='--', linewidth=1.5,
                label=f'Stayed mean ({stayed_yrs.mean():.1f} yr)')
axes[0].axvline(left_yrs.mean(),   color=COLOR_YES, linestyle='--', linewidth=1.5,
                label=f'Left mean ({left_yrs.mean():.1f} yr)')
axes[0].set_title('Years at Company — Stayed vs Left', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Years at Company'); axes[0].set_ylabel('Number of Employees')
axes[0].legend(fontsize=9)

axes[1].hist(df_clean['MonthlyIncome'], bins=30, color='#7c3aed', edgecolor='white', alpha=0.85)
axes[1].axvline(df_clean['MonthlyIncome'].mean(),   color=COLOR_YES, linestyle='--', linewidth=1.5,
                label=f"Mean ${df_clean['MonthlyIncome'].mean():,.0f}")
axes[1].axvline(df_clean['MonthlyIncome'].median(), color=COLOR_NO,  linestyle='--', linewidth=1.5,
                label=f"Median ${df_clean['MonthlyIncome'].median():,.0f}")
axes[1].set_title('Monthly Income Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Monthly Income ($)'); axes[1].set_ylabel('Number of Employees')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Income Group Attrition
ig = attrition_summary(df_clean, 'IncomeGroup')
print(ig.to_string(index=False))

fig, ax = plt.subplots(figsize=(10,5))
bars = ax.bar(ig['IncomeGroup'].astype(str), ig['Rate(%)'],
              color=[COLOR_YES,'#ea580c',COLOR_NO,'#16a34a'],
              edgecolor='white', width=0.55, zorder=3)
add_bar_labels(ax, bars)
avg_line(ax); ax.legend(fontsize=9)
ax.set_title('Attrition Rate by Monthly Income Group', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Income Group'); ax.set_ylabel('Attrition Rate (%)'); ax.set_ylim(0,45)
plt.tight_layout(); plt.show()

---
## 🔟 Correlation Analysis

In [ ]:
corr_cols = ['Age','MonthlyIncome','JobLevel','TotalWorkingYears','YearsAtCompany',
             'YearsInCurrentRole','YearsSinceLastPromotion','YearsWithCurrManager',
             'JobSatisfaction','WorkLifeBalance','AttritionBinary']
corr_labels = ['Age','Monthly\nIncome','Job\nLevel','Total Working\nYears','Years at\nCompany',
               'Years in\nRole','Years Since\nPromotion','Years with\nManager',
               'Job\nSatisfaction','Work-Life\nBalance','Attrition']

corr = df_clean[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(13,10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, ax=ax, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.75, 'label': 'Correlation'},
            xticklabels=corr_labels, yticklabels=corr_labels,
            annot_kws={'size': 8.5})
ax.set_title('Correlation Heatmap — Key Numerical Variables (incl. Attrition binary)',
             fontsize=13, fontweight='bold', pad=15)
plt.tight_layout(); plt.show()

In [ ]:
print("Correlation with Attrition (sorted):")
print(corr['AttritionBinary'].drop('AttritionBinary').sort_values())

> **Strong correlations observed:**  
> - `JobLevel` ↔ `MonthlyIncome` (r ≈ 0.95): Higher seniority = significantly higher pay  
> - `TotalWorkingYears` ↔ `Age` (r ≈ 0.68): More experienced employees are older  
> - **Attrition** correlates negatively with `JobLevel`, `MonthlyIncome`, `Age`, `YearsAtCompany` — confirming junior, lower-paid, less-tenured employees are most at risk  
> - **Note:** Correlation ≠ causation.

---
## 1️⃣1️⃣ Key Insights

In [ ]:
from IPython.display import HTML

insights_data = [
    ('🔴','01','Overtime is the #1 Attrition Driver',
     '30.53% attrition for overtime workers vs 10.44% for non-overtime — nearly 3× higher. '
     '28.3% of all 1,470 employees work overtime.'),
    ('🔴','02','Sales Representatives Are the Highest-Risk Role',
     '39.76% attrition — nearly 2 in 5 leave. Also the lowest-paid role at $2,626/month avg.'),
    ('🟠','03','Young & Entry-Level Employees Leave Most',
     '18–25 age group: 34.78% attrition. Job Level 1: 26.34%. First 0–2 years = highest-risk period.'),
    ('🟠','04','Frequent Travel Triples Attrition Risk',
     '24.91% (frequent) vs 8.00% (non-travel). A 3× difference driven by quality-of-life burden.'),
    ('🟠','05','Poor Work-Life Balance: 31.25% Attrition',
     'WLB Level 1 employees leave at 31.25% — more than double Level 3 (14.22%).'),
    ('🟡','06','$2,046/Month Compensation Gap',
     'Leavers: $4,787/month avg. Stayers: $6,833/month avg. A consistent 30% income difference.'),
    ('🟡','07','Single Employees Leave 2.5× More Than Divorced',
     'Single: 25.53%. Married: 12.48%. Divorced: 10.09%.'),
    ('🟢','08','Job Satisfaction: Higher = Lower Attrition',
     'Level 1 (Low): 22.84% attrition. Level 4 (High): 11.33% attrition.'),
    ('🟢','09','Tenure is Strongly Protective',
     'Attrition declines consistently as years at the company increase.'),
    ('🟢','10','Senior Roles Are Well-Retained',
     'Manager: 4.90%. Research Director: 2.50%. Career progression is a powerful retention mechanism.'),
]

cards = ''.join([
    f'<div style="border-left:5px solid '
    + ('#dc2626' if e=='🔴' else '#d97706' if e=='🟠' else '#ca8a04' if e=='🟡' else '#16a34a')
    + f';background:#f8fafc;padding:12px 16px;margin-bottom:10px;border-radius:6px;">'
    + f'<span style="font-weight:700;font-size:1.05em;color:#1e293b;">{e} [{num}] {title}</span><br>'
    + f'<span style="font-size:0.96em;color:#475569;">{body}</span></div>'
    for e, num, title, body in insights_data
])

display(HTML(f'<div style="font-family:Segoe UI,sans-serif;"><h3 style="color:#1e293b;border-bottom:3px solid #2563eb;padding-bottom:6px;">📋 Key Insights from the Analysis</h3>{cards}</div>'))

---
## 1️⃣2️⃣ Business Recommendations

In [ ]:
from IPython.display import HTML

recs_data = [
    ('R1','Overtime Policies & Monitoring',
     'Based on: 30.53% overtime attrition',
     'Introduce overtime limits and real-time monitoring. Flag high-overtime employees for '
     'engagement check-ins. Redistribute workloads or hire in roles with systemic overtime.'),
    ('R2','Prioritize Sales Retention',
     'Based on: Sales Rep attrition 39.76%; avg $2,626/month',
     'Review base compensation. Introduce performance bonuses, career ladders, mentorship '
     'programs, and manageable travel requirements.'),
    ('R3','Strengthen Early-Career Onboarding',
     'Based on: 18-25 attrition 34.78%; Level 1 attrition 26.34%',
     'Structured 90-day and 12-month onboarding with assigned mentors. Clear 12-24 month '
     'career roadmaps and competitive entry-level pay.'),
    ('R4','Redesign Business Travel Policy',
     'Based on: Frequent travelers 24.91% vs non-travelers 8.00%',
     'Evaluate necessity of all frequent travel. Use virtual alternatives where possible. '
     'Improve allowances and guarantee recovery time after trips.'),
    ('R5','Work-Life Balance Initiatives',
     'Based on: WLB Level 1 attrition 31.25%',
     'Flexible hours, remote work options, wellness programs. Regular pulse surveys '
     'to identify early burnout before it becomes attrition.'),
    ('R6','Compensation Benchmarking',
     'Based on: $2,046/month gap between leavers and stayers',
     'Market salary benchmarking for entry and mid-level roles. Transparent salary '
     'progression milestones and stock option plans.'),
    ('R7','Improve Job Satisfaction',
     'Based on: Level 1 satisfaction → 22.84% attrition vs Level 4 → 11.33%',
     'Quarterly satisfaction surveys with visible follow-up action. Recognition programs '
     'and ensuring employees understand their organizational impact.'),
    ('R8','Create Career Progression Pathways',
     'Based on: Level 1 attrition 26.34% vs Level 4-5 < 7.25%',
     'Individual Development Plans (IDPs), internal-first promotions, and transparent '
     'promotion criteria communicated to all employees.'),
]

rec_cards = ''.join([
    f'<div style="background:#eff6ff;border-left:5px solid #2563eb;padding:12px 16px;margin-bottom:10px;border-radius:6px;">'
    + f'<span style="font-weight:700;color:#1e40af;font-size:1.05em;">[{code}] {title}</span> '
    + f'<span style="font-size:0.88em;color:#64748b;font-style:italic;">— {basis}</span><br>'
    + f'<span style="font-size:0.96em;color:#334155;">{body}</span></div>'
    for code, title, basis, body in recs_data
])

display(HTML(f'<div style="font-family:Segoe UI,sans-serif;"><h3 style="color:#1e293b;border-bottom:3px solid #2563eb;padding-bottom:6px;">💡 Business Recommendations</h3>{rec_cards}</div>'))

---
## 1️⃣3️⃣ Conclusion

In [ ]:
from IPython.display import HTML
display(HTML("""
<div style="background:linear-gradient(135deg,#f0f9ff 0%,#e0f2fe 100%);border:1px solid #bae6fd;
            padding:24px 28px;border-radius:12px;font-family:'Segoe UI',sans-serif;">
  <h3 style="margin:0 0 14px 0;color:#0c4a6e;">✅ Conclusion</h3>
  <p style="color:#1e293b;line-height:1.7;margin:0 0 10px 0;">
    This project applied the complete data analytics lifecycle to the
    <strong>IBM HR Analytics dataset</strong> (1,470 employees, 35 features).
    All findings are grounded in actual calculated values from the dataset.
  </p>
  <p style="color:#1e293b;line-height:1.7;margin:0 0 10px 0;">
    <strong>Key takeaway:</strong> The overall attrition rate is <strong>16.12%</strong>.
    Overtime work (3× higher attrition), low compensation (30% gap), early career stage,
    poor work-life balance, and frequent business travel are the most significant predictors
    of employee departure.
  </p>
  <p style="color:#1e293b;line-height:1.7;margin:0;">
    Eight data-backed recommendations provide HR management with a concrete roadmap to
    measurably reduce attrition through targeted interventions — from overtime policies and
    compensation benchmarking to early onboarding programs and career development pathways.
  </p>
</div>
<p style="font-size:0.88em;color:#94a3b8;margin-top:16px;">
  IBM SkillsBuild Data Analytics with AI Academic Internship Program &nbsp;|&nbsp;
  BharatCares in association with AICTE &nbsp;|&nbsp; Anish Kumar
</p>
"""))